In [7]:
import re
import pickle
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
df = pd.read_csv("../data/raw/Resume.csv")

In [3]:
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [4]:
df = df.drop(["ID", "Resume_html"], axis = 1)

In [5]:
df.head()

,Resume_str,Category
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,HR DIRECTOR Summary Over 2...,HR
3,HR SPECIALIST Summary Dedica...,HR
4,HR MANAGER Skill Highlights ...,HR


In [6]:
df["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
FINANCE                   118
ADVOCATE                  118
ACCOUNTANT                118
ENGINEERING               118
CHEF                      118
AVIATION                  117
FITNESS                   117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [8]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+',        '', text)  # URLs
    text = re.sub(r'[^a-zA-Z\s]',           '', text)  # punctuation
    text = re.sub(r'\s+',                   ' ', text)  # extra spaces
    text = text.strip()
    return text

df['cleaned_resume'] = df['Resume_str'].apply(clean_text)
print("Sample cleaned resume:")
print(df['cleaned_resume'][0][:300])

Sample cleaned resume:
hr administratormarketing associate hr administrator summary dedicated customer service manager with years of experience in hospitality and customer service management respected builder and leader of customerfocused teams strives to instill a shared enthusiastic commitment to customer service highli


In [10]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['Category'])

print("\nLabel mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls} → {i}")

with open("../models/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)


Label mapping:
  ACCOUNTANT → 0
  ADVOCATE → 1
  AGRICULTURE → 2
  APPAREL → 3
  ARTS → 4
  AUTOMOBILE → 5
  AVIATION → 6
  BANKING → 7
  BPO → 8
  BUSINESS-DEVELOPMENT → 9
  CHEF → 10
  CONSTRUCTION → 11
  CONSULTANT → 12
  DESIGNER → 13
  DIGITAL-MEDIA → 14
  ENGINEERING → 15
  FINANCE → 16
  FITNESS → 17
  HEALTHCARE → 18
  HR → 19
  INFORMATION-TECHNOLOGY → 20
  PUBLIC-RELATIONS → 21
  SALES → 22
  TEACHER → 23


In [11]:
VOCAB_SIZE  = 10000
MAX_LENGTH  = 300    

tokenizer = Tokenizer(
    num_words   = VOCAB_SIZE,
    oov_token   = "<OOV>"
)

In [12]:
X = df['cleaned_resume'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 44, stratify = y)

In [13]:
tokenizer.fit_on_texts(X_train)   

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen   = MAX_LENGTH, padding  = 'post', truncating = 'post')
X_test_pad  = pad_sequences(X_test_seq, maxlen   = MAX_LENGTH, padding  = 'post', truncating = 'post')

In [14]:
print("X_train shape:", X_train_pad.shape)
print("X_test shape :", X_test_pad.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (1987, 300)
X_test shape : (497, 300)
y_train shape: (1987,)
y_test shape : (497,)


In [17]:
with open("../models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [18]:
with open("../data/processed/X_train_pad.pkl", "wb") as f:
    pickle.dump(X_train_pad, f)

In [19]:
with open("../data/processed/X_test_pad.pkl", "wb") as f:
    pickle.dump(X_test_pad, f)

In [20]:
with open("../data/processed/y_train.pkl", "wb") as f:
    pickle.dump(y_train, f)

In [21]:
with open("../data/processed/y_test.pkl", "wb") as f:
    pickle.dump(y_test, f)